# B0 / OSNet 256: подтверждение финального результата на трёх seed

Продолжение завершённого `variant_16/review_v1`. Первый final seed **20260915** уже готов и переиспользуется без обучения и пересчёта.
Добавляются только **20260916 и 20260917**, по две модели на seed: B0 и R1_resolution256.
Каждая обучается ровно **800 updates**, LR horizon остаётся **1700**. Итого **4 новых обучения / 3200 updates**.

Все четыре новые модели фиксируются до первой новой outer-оценки. Порог у каждой модели калибруется отдельно только на исходной calibration.
Результат — среднее и разброс по всем трём seed, **не выбор лучшего seed и не ансамбль**.
Решение продолжить принято после просмотра первого outer: это проверка устойчивости на development, не новый независимый тест.

Открой в прежнем `.venv`: **Restart Kernel → Run All**. Старый выполненный notebook не редактируй и не перезапускай одновременно.


In [1]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/final_seed_confirmation.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из проекта Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import torch
from IPython.display import Markdown, display
from training import osnet_review_protocol as review
from training.final_seed_confirmation import prepare, run

print('Repository:', REPO, '| Torch:', torch.__version__)


Repository: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK | Torch: 2.14.0


## 1. Настройки продолжения

Рецепты, seed, устройство и бюджеты наследуются из завершённого manifest, а не задаются заново.
`ALLOW_OUTER_EVALUATION=True` разрешает предусмотренную этой проверкой outer-оценку, но **не замену MVP**.
Не меняй бюджеты, исходники или runtime во время серии. NiVe, новые rerank-параметры и подбор порога по outer не добавляются.


In [2]:
SOURCE_RUN = 'review_v1'
RUN_NAME = 'final_seeds_v1'
ALLOW_OUTER_EVALUATION = True
ALLOW_CPU_TRAINING = False

SOURCE_DIR = review.VARIANT / 'runs' / SOURCE_RUN
FROZEN = review.old.load_json(SOURCE_DIR / 'manifest.json')
DEVICE = FROZEN['runtime']['device']
torch.set_num_threads(FROZEN['runtime']['torch_threads'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(FROZEN['runtime']['deterministic'], warn_only=True)
print('Source:', SOURCE_RUN, '| new run:', RUN_NAME, '| device:', DEVICE)
print('Reused seed:', FROZEN['seeds'][0], '| additional seeds:', FROZEN['seeds'][1:])
print('4 new stages x 800 updates = 3200 updates. MVP unchanged.')


Source: review_v1 | new run: final_seeds_v1 | device: mps
Reused seed: 20260915 | additional seeds: [20260916, 20260917]
4 new stages x 800 updates = 3200 updates. MVP unchanged.


## 2. Проверка исходного финала — без обучения

Проверяются завершённые фазы, исходные веса/отчёты, CSV и изображения, frozen recipes, runtime и fingerprints.
Записывается только новый manifest в variant17. При несовпадении не удаляй старые manifests и не обходи проверку — нужен разбор причины.


In [3]:
context = prepare(RUN_NAME, source_run=SOURCE_RUN)
print('Output:', context['output'])
print('Fingerprint:', context['signature'])
print('Original final preserved:', context['manifest']['extension']['source_directory'])
print('Preflight passed. No optimizer updates yet.')


Output: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_17_final_seed_confirmation/runs/final_seeds_v1
Fingerprint: fe5351deab412917aac0a336af8637dfff1c3ee54230b9058ff936dce8bc4a59
Original final preserved: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_16_review_protocol/runs/review_v1
Preflight passed. No optimizer updates yet.


## 3. Четыре обучения и финальная оценка

Во время обучения выводятся модель, seed, шаг/800, затраченное время и ETA. Во время финальной оценки прогресс может временно не печататься.
После прерывания — **Restart Kernel → Run All** с теми же настройками. Завершённые модели и полностью завершённые оценки проверяются и переиспользуются.
Незавершённый training-блок до 200 updates повторится; если оценка/экспорт были прерваны, повторится эта оценка, а не обучение.

Исходный первый seed не копируется и не оценивается повторно. Его report включается в сводку с той же массой 1/3, что у каждого нового seed.


In [4]:
result = run(context, allow_outer=ALLOW_OUTER_EVALUATION, allow_cpu=ALLOW_CPU_TRAINING)
print('Complete:', result['complete'], '| new updates:', result['new_optimizer_updates'])
print('Promoted:', result['promoted'], '| reused seed:', result['reused_seed'])


final/B0_control/20260916: 200/800 | fixed rule; no selection | elapsed 00:02:58 | ETA 00:08:54
final/B0_control/20260916: 285/800 | fixed rule; no selection | elapsed 00:04:13 | ETA 00:07:38
final/B0_control/20260916: 400/800 | fixed rule; no selection | elapsed 00:05:55 | ETA 00:05:55
final/B0_control/20260916: 600/800 | fixed rule; no selection | elapsed 00:08:52 | ETA 00:02:57
final/B0_control/20260916: 800/800 | fixed rule; no selection | elapsed 00:11:48 | ETA 00:00:00
final/R1_resolution256/20260916: 200/800 | fixed rule; no selection | elapsed 00:03:41 | ETA 00:11:04
final/R1_resolution256/20260916: 285/800 | fixed rule; no selection | elapsed 00:05:15 | ETA 00:09:30
final/R1_resolution256/20260916: 400/800 | fixed rule; no selection | elapsed 00:07:22 | ETA 00:07:22
final/R1_resolution256/20260916: 600/800 | fixed rule; no selection | elapsed 00:11:03 | ETA 00:03:41
final/R1_resolution256/20260916: 800/800 | fixed rule; no selection | elapsed 00:14:45 | ETA 00:00:00
final/B0_c

/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/lib/python3.11/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:2855: UserWarning: ONNX export mode is set to TrainingMode.EVAL, but operator 'instance_norm' is set to train=True. Exporting with train=True.
  symbolic_helper.check_training_mode(use_input_stats, "instance_norm")


Final seed confirmation complete: all three seeds averaged; MVP unchanged.
Complete: True | new updates: 3200
Promoted: False | reused seed: 20260915


## 4. Сводный результат

Для разбора нужны `FINAL_RESULTS.md` и `results.json` в новом run-каталоге.
Отчёт показывает raw/reranked mAP@10, F1/TNR, каждый seed, среднее ± sample SD, paired R1−B0 и masked diagnostics.
Пороги не усредняются. Не выбираем лучший checkpoint/seed по outer, не меняем исходную validation и не продвигаем модель автоматически.
Исторический MVP показан отдельно: он не является четвёртым seed и не входит в средние.


In [5]:
display(Markdown((context['output'] / 'FINAL_RESULTS.md').read_text(encoding='utf-8')))
print('Detailed results:', context['output'] / 'results.json')
print('Preserved first final:', SOURCE_DIR / 'final.json')


# Финальная проверка двух рецептов на трёх seed

Среднее ± выборочное стандартное отклонение по seed; значения в процентах. При неполном наборе значений среднее и SD не вычисляются.

| Модель | Поиск | mAP@10 | F1 | TNR |
|---|---|---:|---:|---:|
| B0_control | raw | 76.714 ± 0.744 (n=3/3) | 74.307 ± 2.255 (n=3/3) | 74.194 ± 8.535 (n=3/3) |
| B0_control | reranked | 77.880 ± 1.278 (n=3/3) | 74.592 ± 2.585 (n=3/3) | 74.194 ± 8.535 (n=3/3) |
| R1_resolution256 | raw | 78.423 ± 0.437 (n=3/3) | 79.121 ± 1.157 (n=3/3) | 67.204 ± 6.715 (n=3/3) |
| R1_resolution256 | reranked | 79.296 ± 0.921 (n=3/3) | 78.570 ± 1.394 (n=3/3) | 67.204 ± 6.715 (n=3/3) |
| MVP, историческая ссылка | raw | 79.016 | 72.549 | 79.032 |
| MVP, историческая ссылка | reranked | 81.469 | 72.861 | 79.032 |

## Парные изменения R1 − B0

Разности в процентных пунктах; seed не выбираются по метрике.

| Поиск | Показатель | Среднее ± SD | 20260915 | 20260916 | 20260917 |
|---|---|---:|---:|---:|---:|
| raw | mAP_at_10 | 1.709 ± 1.099 (n=3/3) | 1.757 | 0.586 | 2.783 |
| raw | candidate_F1 | 4.814 ± 2.466 (n=3/3) | 6.242 | 1.966 | 6.233 |
| raw | TNR | -6.989 ± 1.862 (n=3/3) | -8.065 | -4.839 | -8.065 |
| reranked | mAP_at_10 | 1.415 ± 1.807 (n=3/3) | 1.143 | -0.240 | 3.343 |
| reranked | candidate_F1 | 3.978 ± 3.214 (n=3/3) | 5.364 | 0.304 | 6.267 |
| reranked | TNR | -6.989 ± 1.862 (n=3/3) | -8.065 | -4.839 | -8.065 |

## Каждый seed: original

| Модель | seed | Поиск | mAP@10 | F1 | TNR |
|---|---:|---|---:|---:|---:|
| B0_control | 20260915 | raw | 76.922 | 72.099 | 80.645 |
| B0_control | 20260915 | reranked | 78.752 | 72.414 | 80.645 |
| B0_control | 20260916 | raw | 77.333 | 76.606 | 64.516 |
| B0_control | 20260916 | reranked | 78.475 | 77.449 | 64.516 |
| B0_control | 20260917 | raw | 75.888 | 74.217 | 77.419 |
| B0_control | 20260917 | reranked | 76.414 | 73.913 | 77.419 |
| R1_resolution256 | 20260915 | raw | 78.679 | 78.341 | 72.581 |
| R1_resolution256 | 20260915 | reranked | 79.896 | 77.778 | 72.581 |
| R1_resolution256 | 20260916 | raw | 77.919 | 78.571 | 59.677 |
| R1_resolution256 | 20260916 | reranked | 78.235 | 77.753 | 59.677 |
| R1_resolution256 | 20260917 | raw | 78.672 | 80.449 | 69.355 |
| R1_resolution256 | 20260917 | reranked | 79.757 | 80.180 | 69.355 |

## Автоматические маски: reranked mAP@10

| Условие | B0 | R1 | Парное R1 − B0, п.п. |
|---|---:|---:|---:|
| masked_query | 76.852 ± 1.100 (n=3/3) | 79.024 ± 1.345 (n=3/3) | 2.172 ± 1.750 (n=3/3) |
| masked_gallery | 76.055 ± 1.040 (n=3/3) | 78.601 ± 0.432 (n=3/3) | 2.546 ± 1.270 (n=3/3) |
| masked_both | 74.803 ± 0.698 (n=3/3) | 76.589 ± 0.176 (n=3/3) | 1.787 ± 0.530 (n=3/3) |

## Ограничения

- Это три отдельных обучения, не ансамбль. Нет выбора лучшего seed, усреднения порогов или автозамены MVP.
- Запросы повторяются между seed: их нельзя считать независимыми новыми наблюдениями. SD не является доверительным интервалом.
- Решение добавить два seed принято после просмотра первого outer-результата. Это дополнительная проверка устойчивости на development, не новый независимый тест.
- Исторический MVP выбирался по outer validation; его процедура отбора несимметрична новому протоколу. MVP в таблице не входит в средние или парные разности.
- Маски автоматические и не подтверждены как plate-only. Original validation и исходные bbox не меняются.
- GPU-инференс и официальный скоростной тест не проверены этим отчётом.


Detailed results: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_17_final_seed_confirmation/runs/final_seeds_v1/results.json
Preserved first final: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_16_review_protocol/runs/review_v1/final.json
